# Insulation and peak energy use: a real-world example

This tutorial applies the Systems Insight Pipeline to causal loop diagrams developed in the
[SEVEN project](https://seven.uva.nl/) at the University of Amsterdam, which studies
household energy decisions. See the SEVEN project's report on this work (Fabian Dablander
and colleagues): <https://seven.uva.nl/en/content/news/2026/03/kgg-report.html>.

Two models are included:

- [`Insulation_PeakEnergy.xlsx`](Insulation_PeakEnergy.xlsx) (used here): what drives the
  *percentage of electricity used during peak hours (16.00-21.00)* at the household level —
  31 stocks, 26 constants, 51 candidate interventions, 21 feedback loops.
- [`Insulation_Decisions.xlsx`](Insulation_Decisions.xlsx) (alternate): what drives *the
  number of owner-occupiers who insulate to the standard*. Swap the file name below to
  explore it.

Both files were prepared from raw Kumu exports of the participatory diagrams: variables were
classified as stock/auxiliary/constant, interventions were marked in the `Tags` column, and
the outcome was marked `VOI` in the `Description` column (see
[`Minimal.ipynb`](Minimal.ipynb) for the format).

In [ ]:
from sip_systemsinsightpipeline import Extract, SDM, SDMOptimizer, LoopsThatMatter
from sip_systemsinsightpipeline.plots import plot_simulated_intervention_ranking

extract = Extract("Insulation_PeakEnergy.xlsx")
s = extract.extract_settings()

s.seed = 1912884
s.N = 200                       # parameter samples (increase for smoother estimates)
s.t_end = 15                    # years
s.time_unit = "years"
s.parameter_value_aux = 0.3
s.parameter_value_stocks = 0.1

sdm = SDM(s)
voi = s.variable_of_interest[0]

## Rank interventions under uncertainty

With 51 candidate interventions, the ranking plot is restricted to the strongest 15. Note
that the variable of interest is the share of electricity used during *peak* hours, so
**negative effects are desirable**.

In [ ]:
df_sol_per_sample, param_samples = sdm.run_simulations()
intervention_effects = sdm.get_intervention_effects()

fig = plot_simulated_intervention_ranking(s, intervention_effects[voi], voi, top_plot=15)

## Which links drive the outcome?

The sensitivity analysis correlates each sampled link coefficient with the intervention's
effect on the outcome (Spearman rank correlation, with bootstrapped confidence intervals).
Here we analyze the model's top-ranked intervention.

In [ ]:
top_intervention = list(intervention_effects[voi].keys())[0]
print("Analyzing intervention:", top_intervention)

SA_results, df_SA = sdm.run_SA(voi, top_intervention, cut_off_SA_importance=0.15, n_bootstraps=100)

## Allocate a budget across all interventions

Rather than picking a single intervention, the optimizer splits a budget across all 51
candidates. Because we want to *reduce* peak-hour electricity use, we set
`maximize=False`.

In [ ]:
params = sdm.sample_model_parameters()
costs = [1.0] * len(s.intervention_variables)

optimizer = SDMOptimizer(sdm)
result = optimizer.optimize_intervention_intensities(
    params=params,
    costs=costs,
    variable_of_interest=voi,
    budget=1.0,
    maximize=False,
    n_starts=8,
    seed=1,
)

print("Success:", result["success"])
print("Best effect size:", round(result["best_effect_size"], 4))
print("Near-optimal allocations found:", result["n_equilibria"])

print("\nTop 10 expenditures in the best allocation:")
best = sorted(zip(s.intervention_variables, result["best_y"]), key=lambda kv: -kv[1])
for var, y in best[:10]:
    print(f"  {y:.3f}  {var}")

## How stable is the allocation across parameter uncertainty?

Repeating the optimization for a subset of parameter draws shows which interventions
consistently receive budget. (Each draw runs a 51-dimensional multi-start optimization, so
this cell takes several minutes; increase `n_parameter_samples` for production analyses.)

In [ ]:
opt_df = optimizer.optimize_across_parameter_samples(
    costs=costs,
    variable_of_interest=voi,
    maximize=False,
    n_parameter_samples=10,
    n_starts=4,
    seed=1,
)

intensity_cols = [c for c in opt_df.columns if c.startswith("intensity_")]
mean_alloc = opt_df[intensity_cols].mean().sort_values(ascending=False)
mean_alloc.index = [c.replace("intensity_", "") for c in mean_alloc.index]

ax = mean_alloc.head(10)[::-1].plot(kind="barh", figsize=(8, 5))
ax.set_xlabel("Mean optimal intensity across parameter samples")
ax.set_title("Interventions that consistently receive budget")

## Which feedback loops dominate the dynamics?

The Loops That Matter (LTM) method (Schoenberg, Davidsen & Eberlein) scores every feedback
loop's contribution to the model's behavior at each point in time.

In [ ]:
ltm = LoopsThatMatter(sdm)
ltm.print_loops()

df_sol_ltm, link_scores, loop_scores = ltm.run_ltm_analysis(params, n_points=100)
ltm.get_loop_summary()

## Next steps

- Edit the Excel file: add or remove links, change which variables are tagged as
  interventions, or add a custom `Equation` column entry to make a relationship nonlinear,
  then re-run this notebook.
- See the [README](../README.md) for the full API and Excel format documentation, and the
  D2D paper (Uleman et al., arXiv:2508.05659) for the methodology behind simulating causal
  loop diagrams under uncertainty.